## distilgpt2 and wikitext

In [ ]:
!pip install --quiet transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


ModuleNotFoundError: No module named 'datasets'

In [ ]:
%cd MyDrive/ColabNotebooks/NLP_HW_finetuning

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset

In [ ]:
# 1. Model and tokenizer initialization
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# for models like gpt-2 that don't have a pad token by default
tokenizer.pad_token = tokenizer.eos_token

# for other models without pad token
# tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# 2. Load model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# 3. Load Dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")


train_data = dataset["train"]
val_data = dataset['validation']

# 4. Tokenization Function
def tokenize_function(example):
  result = tokenizer(
                  example["text"],
                  truncation=True,
                  padding="max_length", # could also use ’longest’
                  max_length=512, # set a reasonable max length
                  )
  result["labels"] = result["input_ids"].copy()
  return result


# 5. Apply Tokenization and REmove Original 'text' Column
tokenized_train = train_data.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_val = val_data.map(tokenize_function, batched=True, remove_columns=['text'])

# Remove columns other than input_ids/attention_mask
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
# tokenized_train = tokenized_train.with_format("torch", device=device)
# tokenized_val = tokenized_val.with_format("torch", device=device)
# 7. Training Arguments
training_args = TrainingArguments(
                            output_dir="./distilgpt2+wiki",
                            eval_strategy="epoch",
                            save_strategy="epoch",
                            num_train_epochs=1, # Increase for better results
                            per_device_train_batch_size=8, # Adjust based on GPU memory
                            per_device_eval_batch_size=8,
                            save_steps=500,
                            logging_steps=100,
                            load_best_model_at_end=True,
                            remove_unused_columns=False,
                            push_to_hub=False
                            )

# 8. Trainer Definition
trainer = Trainer(
              model=model,
              args=training_args,
              train_dataset=tokenized_train,
              eval_dataset=tokenized_val,
              tokenizer=tokenizer # tokenizer is important
              )

# 9. Training
trainer.train()

<ipython-input-23-f11ebee62334>:58: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hiteshuv (hiteshuv-university-of-south-florida) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.331800,1.371010


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=4590, training_loss=1.3799982806436377, metrics={'train_runtime': 1008.5375, 'train_samples_per_second': 36.407, 'train_steps_per_second': 4.551, 'total_flos': 1199286761029632.0, 'train_loss': 1.3799982806436377, 'epoch': 1.0})

In [ ]:
prompt = "In␣the␣future,␣we␣wish␣to␣learn␣NLP␣and␣develop␣novel␣artificial␣intelligence␣agents"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
# Generate text
output_ids = model.generate(
                          input_ids,
                          max_length=512,
                          num_beams=5,
                          no_repeat_ngram_size=2,
                          early_stopping=True
                          )
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("Prompt:", prompt)
print("Generated␣text:\n", generated_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt: In␣the␣future,␣we␣wish␣to␣learn␣NLP␣and␣develop␣novel␣artificial␣intelligence␣agents
Generated␣text:
 In␣the␣future,␣we␣wish␣to␣learn␣NLP␣and␣develop␣novel␣artificial␣intelligence␣agents.

This article was originally published on The Conversation. Read the original article.


In [ ]:
model.save_pretrained("./models/distilgpt2-wiki")
tokenizer.save_pretrained("./models/distilgpt2-wiki")